# 🧑‍💻 Reconhecimento Facial com YOLO + TensorFlow no Google Colab

## Arquitetura do pipeline

```
Imagem → YOLO (detecta rostos) → Recorta cada rosto → MobileNetV2 (classifica quem é) → Resultado
```

- **Estágio 1 — Detecção:** YOLOv8-face localiza todos os rostos na imagem (bounding boxes)
- **Estágio 2 — Reconhecimento:** MobileNetV2 (TensorFlow/Keras) com transfer learning classifica cada rosto recortado

## Classes de reconhecimento
- `eu` → fotos suas (mínimo 15–20, idealmente 50+)
- `desconhecido` → fotos de outras pessoas (rostos variados)

## ⚙️ Antes de começar
`Ambiente de execução → Alterar tipo de ambiente → GPU (T4)`

---
## 1. Setup

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q ultralytics

import ultralytics
ultralytics.checks()

import tensorflow as tf
print(f'TensorFlow: {tf.__version__}')
print(f'GPU disponível: {tf.config.list_physical_devices("GPU")}')

In [ ]:
from pathlib import Path
import os, shutil

WORK = Path('/content/face_recognition')
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
print(f'Pasta de trabalho: {WORK}')

---
## 2. Montar dataset de rostos

Você precisa de fotos organizadas assim:
```
dataset_faces/
├── eu/              ← 15–50 fotos do seu rosto (selfies, ângulos variados)
└── desconhecido/    ← 30–100 fotos de outras pessoas
```

**Execute UMA das opções abaixo:**

- **Opção A** — Upload manual (poucos arquivos)
- **Opção B** — Google Drive (recomendado)
- **Opção C** — Upload de ZIP

### 🅰️ Opção A — Upload manual de fotos

A célula abre duas janelas de upload:
1. Primeiro envie suas fotos (classe `eu`)
2. Depois envie fotos de outras pessoas (classe `desconhecido`)

In [ ]:
# ===== OPÇÃO A: Upload manual =====
from google.colab import files
from pathlib import Path

DS = Path('/content/face_recognition/dataset_faces')
for cls in ['eu', 'desconhecido']:
    d = DS / cls
    d.mkdir(parents=True, exist_ok=True)

# --- Suas fotos ---
print('📸 Envie suas fotos (classe "eu"):')
uploaded_eu = files.upload()
for name, data in uploaded_eu.items():
    (DS / 'eu' / name).write_bytes(data)
print(f'✅ {len(uploaded_eu)} fotos salvas em dataset_faces/eu/')

# --- Fotos de outras pessoas ---
print('\n📸 Envie fotos de outras pessoas (classe "desconhecido"):')
uploaded_other = files.upload()
for name, data in uploaded_other.items():
    (DS / 'desconhecido' / name).write_bytes(data)
print(f'✅ {len(uploaded_other)} fotos salvas em dataset_faces/desconhecido/')

### 🅱️ Opção B — Google Drive

In [ ]:
# ===== OPÇÃO B: Google Drive =====
from google.colab import drive
drive.mount('/content/drive')

# Ajuste o caminho para sua pasta no Drive
DRIVE_PATH = '/content/drive/MyDrive/dataset_faces'

!cp -r "$DRIVE_PATH" /content/face_recognition/dataset_faces
print('✅ Dataset copiado do Drive')

### 🅲 Opção C — Upload de ZIP

In [ ]:
# ===== OPÇÃO C: Upload de ZIP =====
from google.colab import files
import zipfile

print('Envie o dataset_faces.zip...')
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_name) as z:
    z.extractall('/content/face_recognition/')

print(f'✅ {zip_name} extraído')

### Verificar dataset

In [ ]:
from pathlib import Path

DS = Path('/content/face_recognition/dataset_faces')
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

print('=== Dataset de rostos ===')
for cls_dir in sorted(DS.iterdir()):
    if cls_dir.is_dir():
        imgs = [f for f in cls_dir.iterdir() if f.suffix.lower() in IMG_EXTS]
        print(f'  📁 {cls_dir.name}: {len(imgs)} imagens')

assert (DS / 'eu').exists(), '❌ Pasta dataset_faces/eu/ não encontrada!'
assert (DS / 'desconhecido').exists(), '❌ Pasta dataset_faces/desconhecido/ não encontrada!'

---
## 3. Extrair rostos com YOLO (Estágio 1)

Usamos um modelo YOLO treinado para detecção de rostos.
Ele localiza cada rosto, recorta e salva em pastas separadas.
Essas imagens recortadas alimentam o classificador TensorFlow.

In [ ]:
import cv2
import numpy as np
from pathlib import Path
from ultralytics import YOLO

# --- Modelo YOLO para detecção de rostos ---
# yolov8n-face é um modelo fine-tuned para rostos.
# Se não estiver disponível, usamos yolov8n normal (detecta "person")
# e aplicamos um crop na região superior do corpo.

# Tenta baixar o modelo de faces do Ultralytics HUB
FACE_MODEL_URL = 'https://github.com/akanametov/yolov8-face/releases/download/v0.0.0/yolov8n-face.pt'
FACE_MODEL_PATH = Path('/content/face_recognition/yolov8n-face.pt')

if not FACE_MODEL_PATH.exists():
    print('Baixando modelo YOLOv8n-face...')
    import urllib.request
    try:
        urllib.request.urlretrieve(FACE_MODEL_URL, FACE_MODEL_PATH)
        print('✅ yolov8n-face.pt baixado')
    except Exception as e:
        print(f'⚠️ Falha ao baixar yolov8n-face: {e}')
        print('Usando yolov8n.pt padrão (detecta pessoas)')
        FACE_MODEL_PATH = None

In [ ]:
import cv2
import numpy as np
from pathlib import Path
from ultralytics import YOLO

# Carrega o modelo de detecção de rostos
if FACE_MODEL_PATH and FACE_MODEL_PATH.exists():
    face_detector = YOLO(str(FACE_MODEL_PATH))
    FACE_CLASS = None  # yolov8n-face tem só uma classe (face)
    print('Usando: yolov8n-face (detecção específica de rostos)')
else:
    face_detector = YOLO('yolov8n.pt')
    FACE_CLASS = 0  # classe 'person' no COCO
    print('Usando: yolov8n (detecção de pessoas — fará crop da região da cabeça)')

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
FACE_SIZE = (160, 160)  # tamanho padrão para o classificador


def extract_faces_from_image(img_path, detector, face_class=None,
                              conf=0.3, margin=0.2):
    """
    Detecta rostos na imagem e retorna lista de crops (BGR).
    margin: % extra ao redor do rosto para pegar mais contexto.
    """
    img = cv2.imread(str(img_path))
    if img is None:
        return []

    h, w = img.shape[:2]
    results = detector.predict(source=img, conf=conf, verbose=False)
    faces = []

    for r in results:
        if r.boxes is None:
            continue
        for box in r.boxes:
            cls_id = int(box.cls[0])
            # Se é yolov8n padrão, só pega classe 'person' e recorta cabeça
            if face_class is not None and cls_id != face_class:
                continue

            x1, y1, x2, y2 = box.xyxy[0].tolist()

            # Se usando detector de pessoa, estima a cabeça (top 30%)
            if face_class is not None:
                body_h = y2 - y1
                y2 = y1 + body_h * 0.30
                cx = (x1 + x2) / 2
                head_w = (x2 - x1) * 0.6
                x1 = cx - head_w / 2
                x2 = cx + head_w / 2

            # Adiciona margem
            bw, bh = x2 - x1, y2 - y1
            x1 = max(0, int(x1 - bw * margin))
            y1 = max(0, int(y1 - bh * margin))
            x2 = min(w, int(x2 + bw * margin))
            y2 = min(h, int(y2 + bh * margin))

            crop = img[y1:y2, x1:x2]
            if crop.size > 0 and crop.shape[0] > 10 and crop.shape[1] > 10:
                crop = cv2.resize(crop, FACE_SIZE)
                faces.append(crop)

    return faces


def extract_faces_dataset(input_dir, output_dir, detector, face_class=None):
    """
    Para cada classe (eu, desconhecido), extrai rostos de todas as imagens
    e salva como novos arquivos.
    """
    output_dir = Path(output_dir)
    stats = {}

    for cls_dir in sorted(Path(input_dir).iterdir()):
        if not cls_dir.is_dir():
            continue

        cls_name = cls_dir.name
        out_cls = output_dir / cls_name
        out_cls.mkdir(parents=True, exist_ok=True)

        imgs = [f for f in cls_dir.iterdir() if f.suffix.lower() in IMG_EXTS]
        count = 0

        for img_path in imgs:
            faces = extract_faces_from_image(img_path, detector, face_class)
            if not faces:
                # Se YOLO não detectou, salva a imagem inteira redimensionada
                img = cv2.imread(str(img_path))
                if img is not None:
                    img = cv2.resize(img, FACE_SIZE)
                    faces = [img]

            for i, face in enumerate(faces):
                out_path = out_cls / f'{img_path.stem}_face{i}.jpg'
                cv2.imwrite(str(out_path), face)
                count += 1

        stats[cls_name] = count
        print(f'  📁 {cls_name}: {count} rostos extraídos de {len(imgs)} imagens')

    return stats


# --- Executar extração ---
FACES_DIR = Path('/content/face_recognition/faces_extracted')
if FACES_DIR.exists():
    shutil.rmtree(FACES_DIR)

print('Extraindo rostos com YOLO...\n')
stats = extract_faces_dataset(
    '/content/face_recognition/dataset_faces',
    FACES_DIR,
    face_detector,
    FACE_CLASS,
)
print(f'\n✅ Total de rostos extraídos: {sum(stats.values())}')

In [ ]:
# Visualizar exemplos de rostos extraídos
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('Rostos extraídos por YOLO', fontsize=14)

for row, cls in enumerate(['eu', 'desconhecido']):
    cls_dir = FACES_DIR / cls
    imgs = sorted(cls_dir.glob('*.jpg'))[:5]
    for col in range(5):
        ax = axes[row][col]
        if col < len(imgs):
            img = cv2.imread(str(imgs[col]))
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            if col == 0:
                ax.set_ylabel(cls, fontsize=12, fontweight='bold')
        ax.axis('off')

plt.tight_layout()
plt.show()

---
## 4. Treinar classificador com TensorFlow (Estágio 2)

Usamos **MobileNetV2** (transfer learning) para classificar rostos.

**Arquitetura:**
```
MobileNetV2 (backbone congelado, pesos ImageNet)
  → GlobalAveragePooling2D
  → Dropout(0.3)
  → Dense(128, relu)
  → Dropout(0.3)
  → Dense(num_classes, softmax)
```

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator

FACES_DIR = Path('/content/face_recognition/faces_extracted')
IMG_SIZE = (160, 160)
BATCH_SIZE = 16

# --- Data Augmentation e split treino/validação ---
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    validation_split=0.2,       # 80% treino, 20% validação
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    horizontal_flip=True,
    brightness_range=[0.7, 1.3],
    zoom_range=0.15,
    fill_mode='nearest',
)

val_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    validation_split=0.2,
)

train_gen = train_datagen.flow_from_directory(
    FACES_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=42,
)

val_gen = val_datagen.flow_from_directory(
    FACES_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=42,
)

CLASS_NAMES = list(train_gen.class_indices.keys())
NUM_CLASSES = len(CLASS_NAMES)

print(f'\nClasses: {CLASS_NAMES}')
print(f'Amostras treino:    {train_gen.samples}')
print(f'Amostras validação: {val_gen.samples}')

In [ ]:
# --- Montar o modelo com MobileNetV2 + Transfer Learning ---

# Backbone pré-treinado no ImageNet (congelado)
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(*IMG_SIZE, 3),
    include_top=False,
    weights='imagenet',
)
base_model.trainable = False  # congela backbone

# Cabeça de classificação
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation='softmax'),
], name='face_classifier')

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

model.summary()

In [ ]:
# --- Fase 1: treinar só a cabeça (backbone congelado) ---

print('=== Fase 1: treinar cabeça (backbone congelado) ===')

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True
    ),
]

history1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=20,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
# --- Fase 2: Fine-tuning — descongelar últimas camadas do backbone ---

print('=== Fase 2: Fine-tuning (últimas 30 camadas descongeladas) ===')

base_model.trainable = True
# Congela tudo exceto as últimas 30 camadas
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Recompila com learning rate menor
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

trainable_count = sum(1 for l in model.layers for w in l.weights if w.trainable)
print(f'Parâmetros treináveis: {sum(w.numpy().size for w in model.trainable_weights):,}')

history2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=15,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
# --- Gráficos de treino ---
import matplotlib.pyplot as plt

def plot_history(h1, h2):
    acc = h1.history['accuracy'] + h2.history['accuracy']
    val_acc = h1.history['val_accuracy'] + h2.history['val_accuracy']
    loss = h1.history['loss'] + h2.history['loss']
    val_loss = h1.history['val_loss'] + h2.history['val_loss']
    epochs = range(1, len(acc) + 1)
    phase2_start = len(h1.history['accuracy'])

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(epochs, acc, 'b-', label='Treino')
    ax1.plot(epochs, val_acc, 'r-', label='Validação')
    ax1.axvline(x=phase2_start, color='gray', linestyle='--', label='Fine-tuning')
    ax1.set_title('Acurácia')
    ax1.set_xlabel('Época'); ax1.set_ylabel('Acurácia')
    ax1.legend()

    ax2.plot(epochs, loss, 'b-', label='Treino')
    ax2.plot(epochs, val_loss, 'r-', label='Validação')
    ax2.axvline(x=phase2_start, color='gray', linestyle='--', label='Fine-tuning')
    ax2.set_title('Loss')
    ax2.set_xlabel('Época'); ax2.set_ylabel('Loss')
    ax2.legend()

    plt.tight_layout()
    plt.show()

plot_history(history1, history2)

In [ ]:
# Salvar modelo
MODEL_PATH = Path('/content/face_recognition/face_classifier.keras')
model.save(MODEL_PATH)
print(f'✅ Modelo salvo: {MODEL_PATH} ({MODEL_PATH.stat().st_size/1e6:.1f} MB)')

---
## 5. Pipeline completo — YOLO detecta + TensorFlow classifica

Agora juntamos os dois estágios:
1. YOLO encontra os rostos na imagem
2. TensorFlow classifica cada rosto
3. Desenhamos bounding boxes com o nome + confiança

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from ultralytics import YOLO
from pathlib import Path

# Carregar modelos
FACE_MODEL_PATH = Path('/content/face_recognition/yolov8n-face.pt')
if FACE_MODEL_PATH.exists():
    yolo_face = YOLO(str(FACE_MODEL_PATH))
    USE_PERSON_CROP = False
else:
    yolo_face = YOLO('yolov8n.pt')
    USE_PERSON_CROP = True

classifier = tf.keras.models.load_model('/content/face_recognition/face_classifier.keras')
CLASS_NAMES = sorted([d.name for d in Path('/content/face_recognition/faces_extracted').iterdir() if d.is_dir()])
print(f'Classes: {CLASS_NAMES}')

FACE_SIZE = (160, 160)


def recognize_faces(img_path, conf_yolo=0.3, conf_cls=0.6):
    """
    Pipeline completo:
    1. YOLO detecta rostos
    2. TensorFlow classifica cada rosto
    3. Retorna imagem com bounding boxes e rótulos
    """
    img = cv2.imread(str(img_path))
    if img is None:
        raise FileNotFoundError(f'Não foi possível abrir: {img_path}')

    display = img.copy()
    h, w = img.shape[:2]

    # Detecção YOLO
    results = yolo_face.predict(source=img, conf=conf_yolo, verbose=False)

    detections = []
    for r in results:
        if r.boxes is None:
            continue
        for box in r.boxes:
            cls_id = int(box.cls[0])
            if USE_PERSON_CROP and cls_id != 0:
                continue

            x1, y1, x2, y2 = box.xyxy[0].tolist()

            # Se detector de pessoa, estima cabeça
            if USE_PERSON_CROP:
                body_h = y2 - y1
                y2 = y1 + body_h * 0.30
                cx = (x1 + x2) / 2
                head_w = (x2 - x1) * 0.6
                x1 = cx - head_w / 2
                x2 = cx + head_w / 2

            # Margem
            margin = 0.2
            bw, bh = x2 - x1, y2 - y1
            x1m = max(0, int(x1 - bw * margin))
            y1m = max(0, int(y1 - bh * margin))
            x2m = min(w, int(x2 + bw * margin))
            y2m = min(h, int(y2 + bh * margin))

            crop = img[y1m:y2m, x1m:x2m]
            if crop.size == 0 or crop.shape[0] < 10 or crop.shape[1] < 10:
                continue

            # Classificação com TensorFlow
            face_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
            face_resized = cv2.resize(face_rgb, FACE_SIZE)
            face_input = face_resized.astype('float32') / 255.0
            face_input = np.expand_dims(face_input, axis=0)

            preds = classifier.predict(face_input, verbose=0)[0]
            class_idx = np.argmax(preds)
            confidence = preds[class_idx]
            label = CLASS_NAMES[class_idx]

            # Se confiança baixa, marca como desconhecido
            if confidence < conf_cls:
                label = 'desconhecido'
                confidence = 1.0 - preds[class_idx]

            detections.append({
                'bbox': (x1m, y1m, x2m, y2m),
                'label': label,
                'confidence': confidence,
            })

            # Desenhar na imagem
            color = (0, 255, 0) if label == 'eu' else (0, 0, 255)
            cv2.rectangle(display, (x1m, y1m), (x2m, y2m), color, 3)

            text = f'{label} {confidence:.0%}'
            (tw, th), bl = cv2.getTextSize(
                text, cv2.FONT_HERSHEY_SIMPLEX, 0.8, 2
            )
            cv2.rectangle(
                display,
                (x1m, y1m - th - bl - 8),
                (x1m + tw + 8, y1m),
                color, -1,
            )
            cv2.putText(
                display, text,
                (x1m + 4, y1m - bl - 4),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8, (255, 255, 255), 2, cv2.LINE_AA,
            )

    return display, detections


print('✅ Pipeline carregado')

### 5.1 Testar com imagem do dataset

In [ ]:
# Escolhe uma imagem da pasta "eu" para testar
DS = Path('/content/face_recognition/dataset_faces')
test_imgs = list((DS / 'eu').glob('*.jpg')) + list((DS / 'eu').glob('*.png'))

if test_imgs:
    TEST_IMG = str(test_imgs[0])
    print(f'Testando com: {TEST_IMG}')

    result_img, dets = recognize_faces(TEST_IMG)

    print(f'\n{len(dets)} rosto(s) detectado(s):')
    for d in dets:
        print(f'  {d["label"]:15s} confiança={d["confidence"]:.1%}')

    plt.figure(figsize=(10, 8))
    plt.imshow(cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB))
    plt.title('Reconhecimento: YOLO (detecção) + TensorFlow (classificação)')
    plt.axis('off')
    plt.show()
else:
    print('⚠️ Nenhuma imagem encontrada em dataset_faces/eu/')

### 5.2 Testar com upload de nova foto

In [ ]:
from google.colab import files

print('📸 Envie uma foto para testar o reconhecimento:')
uploaded = files.upload()

for name in uploaded:
    img_path = f'/content/{name}'
    with open(img_path, 'wb') as f:
        f.write(uploaded[name])

    result_img, dets = recognize_faces(img_path)

    print(f'\n{len(dets)} rosto(s) detectado(s) em {name}:')
    for d in dets:
        print(f'  {d["label"]:15s} confiança={d["confidence"]:.1%}')

    plt.figure(figsize=(12, 9))
    plt.imshow(cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB))
    plt.title(f'Resultado: {name}')
    plt.axis('off')
    plt.show()

### 5.3 Testar com webcam (captura pelo Colab)

In [ ]:
from IPython.display import display, Javascript, Image as IPImage
from google.colab.output import eval_js
import base64, io

def capture_webcam():
    js = Javascript('''
    async function capture() {
        const div = document.createElement('div');
        const video = document.createElement('video');
        const btn = document.createElement('button');
        btn.textContent = '📸 Capturar';
        btn.style.fontSize = '18px';
        btn.style.padding = '10px 20px';
        btn.style.margin = '10px';
        div.appendChild(video);
        div.appendChild(document.createElement('br'));
        div.appendChild(btn);
        document.body.appendChild(div);

        const stream = await navigator.mediaDevices.getUserMedia({video: true});
        video.srcObject = stream;
        await video.play();

        // Espera clicar
        await new Promise(r => btn.onclick = r);

        const canvas = document.createElement('canvas');
        canvas.width = video.videoWidth;
        canvas.height = video.videoHeight;
        canvas.getContext('2d').drawImage(video, 0, 0);

        stream.getTracks().forEach(t => t.stop());
        div.remove();
        return canvas.toDataURL('image/jpeg', 0.9);
    }
    ''')
    display(js)
    data = eval_js('capture()')
    binary = base64.b64decode(data.split(',')[1])
    return binary

print('Clique em "Capturar" quando estiver pronto...')
img_bytes = capture_webcam()

webcam_path = '/content/webcam_capture.jpg'
with open(webcam_path, 'wb') as f:
    f.write(img_bytes)

result_img, dets = recognize_faces(webcam_path)

print(f'\n{len(dets)} rosto(s) detectado(s):')
for d in dets:
    print(f'  {d["label"]:15s} confiança={d["confidence"]:.1%}')

plt.figure(figsize=(10, 8))
plt.imshow(cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB))
plt.title('Webcam — Reconhecimento facial')
plt.axis('off')
plt.show()

---
## 6. Salvar tudo no Google Drive

In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')

    drive_out = Path('/content/drive/MyDrive/face_recognition_results')
    drive_out.mkdir(exist_ok=True)

    import shutil
    # Modelo TensorFlow
    shutil.copy('/content/face_recognition/face_classifier.keras',
                drive_out / 'face_classifier.keras')

    # Modelo YOLO face (se existir)
    yolo_face_pt = Path('/content/face_recognition/yolov8n-face.pt')
    if yolo_face_pt.exists():
        shutil.copy(yolo_face_pt, drive_out / 'yolov8n-face.pt')

    # Salvar lista de classes
    (drive_out / 'classes.txt').write_text('\n'.join(CLASS_NAMES))

    print(f'\n✅ Modelos salvos em: {drive_out}')
    for p in drive_out.iterdir():
        print(f'  {p.name} ({p.stat().st_size/1e6:.1f} MB)')

except Exception as e:
    print(f'⚠️  {e}')
    print('Baixe manualmente:')
    from google.colab import files
    files.download('/content/face_recognition/face_classifier.keras')

---
## ✅ Resumo do que foi feito

| Componente | Framework | Função |
|---|---|---|
| **Detecção de rostos** | YOLO (Ultralytics) | Localiza rostos na imagem (bounding boxes) |
| **Classificação facial** | TensorFlow/Keras (MobileNetV2) | Identifica de quem é o rosto (transfer learning) |
| **Pipeline combinado** | YOLO + TensorFlow | Detecta → recorta → classifica → desenha resultado |

### 💡 Dicas para melhorar a acurácia
- Mais fotos suas (50+), com variação de luz, ângulo e expressão
- Mais fotos de desconhecidos (100+), com diversidade de rostos
- Para adicionar mais pessoas, crie pastas adicionais: `dataset_faces/joao/`, `dataset_faces/maria/` etc.